# Install Dependencies

In [1]:
!pip install transformers torch

# Import Libraries

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load Pre-trained Model

In [14]:
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
chat_history_ids = None

print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! 👋")
        break

    # Better prompt
    user_input = "User: " + user_input + " Bot:"

    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
    new_attention_mask = torch.ones_like(new_input_ids)

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        attention_mask = torch.cat([torch.ones_like(chat_history_ids), new_attention_mask], dim=-1)
    else:
        bot_input_ids = new_input_ids
        attention_mask = new_attention_mask

    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=200,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.7,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2
    )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.
You: Hello
Chatbot: I'm not a bot.
You: What is an ai bot?
Chatbot: User : What is a? Bot : A bot that posts comments on the internet, but also posts replies and links to other bots.
You: Why are you dumb bro?
Chatbot: user : No u
You: exit
Chatbot: Goodbye! 👋
